# Advanced Problems with Solutions: Function Attributes, Bound Methods, and Method Binding

This notebook is an advanced practice set based on the topic of **functions defined in classes becoming bound methods when accessed through instances**.

## Learning goals

By the end, you should be able to reason precisely about:

- functions stored in a class `__dict__`
- instance method binding
- the implicit first argument conventionally named `self`
- the difference between `Class.method` and `instance.method`
- `method.__func__` and `method.__self__`
- repeated bound-method lookup
- calling an instance method through the class
- monkey-patching classes at runtime
- assigning ordinary functions directly to an instance
- manually binding functions with `types.MethodType`
- the function descriptor protocol via `__get__`
- method shadowing by instance attributes
- callbacks that retain bound instances
- inheritance and method binding
- robust tests for method identity and behavior

The notebook intentionally contains **many examples, problems, and lines of code**.  
Run the cells from top to bottom.


## 0. Baseline mental model

A function defined in a class body is stored as a function object in the class namespace.

```python
class Person:
    def hello(self):
        return "hello"
```

Conceptually:

- `Person.__dict__["hello"]` is a function.
- `Person.hello` is also accessed as a function from the class.
- `p.hello` is a **bound method** when `p` is a `Person`.
- that bound method remembers:
  - the original function in `.__func__`
  - the bound object in `.__self__`

Calling:

```python
p.hello()
```

is behaviorally similar to:

```python
Person.hello(p)
```

The first form uses automatic binding; the second passes the instance explicitly.


In [1]:
# Baseline setup

class Person:
    def __init__(self, name):
        self.name = name

    def hello(self):
        return f"Hello from {self.name}"

p = Person("Ada")

print("Person.__dict__['hello']:", Person.__dict__["hello"])
print("Person.hello:", Person.hello)
print("p.hello:", p.hello)

print("type(Person.hello):", type(Person.hello))
print("type(p.hello):", type(p.hello))

print("p.hello.__func__ is Person.hello:",
      p.hello.__func__ is Person.hello)
print("p.hello.__self__ is p:",
      p.hello.__self__ is p)

print("p.hello():", p.hello())
print("Person.hello(p):", Person.hello(p))


Person.__dict__['hello']: <function Person.hello at 0x000002C81C780F40>
Person.hello: <function Person.hello at 0x000002C81C780F40>
p.hello: <bound method Person.hello of <__main__.Person object at 0x000002C80C6F4440>>
type(Person.hello): <class 'function'>
type(p.hello): <class 'method'>
p.hello.__func__ is Person.hello: True
p.hello.__self__ is p: True
p.hello(): Hello from Ada
Person.hello(p): Hello from Ada


# Problem 1 — Predict exact binding behavior

Without running the next cell first, predict:

1. the type of `Worker.run`
2. the type of `w.run`
3. what object `w.run.__self__` references
4. whether `w.run.__func__ is Worker.run`
5. whether `w.run(3)` and `Worker.run(w, 3)` produce the same result


In [2]:
class Worker:
    def __init__(self, rate):
        self.rate = rate

    def run(self, hours):
        return self.rate * hours

w = Worker(25)

# Write your predictions as comments before running:
# 1.
# 2.
# 3.
# 4.
# 5.


## Solution 1


In [3]:
assert type(Worker.run).__name__ == "function"
assert type(w.run).__name__ == "method"
assert w.run.__self__ is w
assert w.run.__func__ is Worker.run
assert w.run(3) == Worker.run(w, 3) == 75

print("type(Worker.run):", type(Worker.run))
print("type(w.run):", type(w.run))
print("w.run.__self__ is w:", w.run.__self__ is w)
print("w.run.__func__ is Worker.run:", w.run.__func__ is Worker.run)
print("w.run(3):", w.run(3))
print("Worker.run(w, 3):", Worker.run(w, 3))


type(Worker.run): <class 'function'>
type(w.run): <class 'method'>
w.run.__self__ is w: True
w.run.__func__ is Worker.run: True
w.run(3): 75
Worker.run(w, 3): 75


# Problem 2 — Why does the missing `self` error happen?

The following class defines a zero-argument function inside a class.

Predict what happens for:

```python
Broken.ping()
Broken().ping()
```

Explain *why* the two calls behave differently.


In [4]:
class Broken:
    def ping():
        return "pong"

print("Calling through the class:")
print(Broken.ping())

print("\nCalling through an instance:")
try:
    print(Broken().ping())
except TypeError as ex:
    print(type(ex).__name__ + ":", ex)


Calling through the class:
pong

Calling through an instance:
TypeError: Broken.ping() takes 0 positional arguments but 1 was given


## Solution 2

`Broken.ping()` accesses the function through the class, so no instance is automatically supplied.

`Broken().ping()` accesses the function through an instance. Python creates a bound method and supplies the instance as the first positional argument. The underlying function declared zero parameters, so the call receives one positional argument too many.


In [5]:
b = Broken()

bound = b.ping

print("Underlying function:", bound.__func__)
print("Bound object:", bound.__self__)
print("Underlying function parameter mismatch demonstrated below:")

try:
    bound.__func__(b)
except TypeError as ex:
    print(type(ex).__name__ + ":", ex)


Underlying function: <function Broken.ping at 0x000002C81C76CF40>
Bound object: <__main__.Broken object at 0x000002C81C774690>
Underlying function parameter mismatch demonstrated below:
TypeError: Broken.ping() takes 0 positional arguments but 1 was given


# Problem 3 — `self` is a convention, not syntax

Refactor the class below so that the method still works **without** using the name `self`.

Then prove that Python binds the instance regardless of the first parameter's name.


In [6]:
class Counter:
    def __init__(instance, start=0):
        instance.value = start

    def increment(current_object, amount=1):
        current_object.value += amount
        return current_object.value

c = Counter(10)

print(c.increment())
print(c.increment(5))
print(c.value)

assert c.value == 16


11
16
16


## Solution 3

The first parameter can have any valid parameter name. `self` is the standard convention because it makes code immediately recognizable to Python programmers.

Python's binding behavior is not based on the spelling `self`; it is based on accessing a function descriptor through an instance.


In [7]:
m = c.increment

print("Bound instance:", m.__self__)
print("Underlying function:", m.__func__)
print("Function parameter names:", m.__func__.__code__.co_varnames[:m.__func__.__code__.co_argcount])

assert m.__self__ is c
assert m.__func__ is Counter.increment


Bound instance: <__main__.Counter object at 0x000002C81C6DEBA0>
Underlying function: <function Counter.increment at 0x000002C81C780B80>
Function parameter names: ('current_object', 'amount')


# Problem 4 — Are repeated bound-method lookups the same object?

Predict the result of:

```python
p.hello is p.hello
```

Then compare it with:

```python
p.hello == p.hello
```

Finally, store a bound method once and compare the stored reference with itself.


In [8]:
class Greeter:
    def greet(self):
        return "hi"

g = Greeter()

a = g.greet
b = g.greet

print("a is b:", a is b)
print("a == b:", a == b)
print("a.__func__ is b.__func__:", a.__func__ is b.__func__)
print("a.__self__ is b.__self__:", a.__self__ is b.__self__)
print("a is a:", a is a)


a is b: False
a == b: True
a.__func__ is b.__func__: True
a.__self__ is b.__self__: True
a is a: True


## Solution 4

A fresh bound-method object is generally created on each attribute access, so identity with `is` is not the right way to test whether two method lookups represent the same binding.

The useful structural checks are:

```python
a.__func__ is b.__func__
a.__self__ is b.__self__
```

A stored reference such as `a` is, of course, identical to itself.


In [9]:
assert a.__func__ is b.__func__
assert a.__self__ is b.__self__

# Avoid relying on object identity for repeated method lookup:
assert a is a

print("Structural identity confirmed.")


Structural identity confirmed.


# Problem 5 — Extract and call the underlying function

Given a bound method `account.deposit`, recover its underlying function and bound instance, then reproduce the method call manually.

Do not use `Account.deposit` directly in your manual call.


In [10]:
class Account:
    def __init__(self, balance=0):
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount
        return self.balance

account = Account(100)
method = account.deposit

# TODO:
# function = ...
# instance = ...
# result = function(instance, 50)


## Solution 5


In [11]:
function = method.__func__
instance = method.__self__

result = function(instance, 50)

print("function:", function)
print("instance:", instance)
print("result:", result)
print("balance:", account.balance)

assert function is Account.deposit
assert instance is account
assert result == 150
assert account.balance == 150


function: <function Account.deposit at 0x000002C81C780C20>
instance: <__main__.Account object at 0x000002C81C6DEE40>
result: 150
balance: 150


# Problem 6 — Class monkey-patching becomes method binding

Add a function to an existing class at runtime, then show that:

- the class stores a function
- an instance exposes a bound method
- `__func__` points to the monkey-patched function
- the function receives the instance automatically


In [12]:
class Device:
    def __init__(self, name):
        self.name = name

def describe(self):
    return f"Device<{self.name}>"

Device.describe = describe

d = Device("sensor-7")

print("Class dictionary object:", Device.__dict__["describe"])
print("Via class:", Device.describe)
print("Via instance:", d.describe)
print("Call:", d.describe())


Class dictionary object: <function describe at 0x000002C81C780CC0>
Via class: <function describe at 0x000002C81C780CC0>
Via instance: <bound method describe of <__main__.Device object at 0x000002C81C6DEA50>>
Call: Device<sensor-7>


## Solution 6


In [13]:
assert Device.__dict__["describe"] is describe
assert Device.describe is describe
assert d.describe.__func__ is describe
assert d.describe.__self__ is d
assert d.describe() == "Device<sensor-7>"

print("Monkey-patched class function binds normally.")


Monkey-patched class function binds normally.


# Problem 7 — Instance monkey-patching does NOT automatically bind

Assign a plain function directly to an instance and inspect what happens.

Then fix it so that the function behaves like an instance method.


In [14]:
class Robot:
    def __init__(self, name):
        self.name = name

r = Robot("R2")

def report(self, value):
    return f"{self.name}: {value}"

r.report = report

print("r.report:", r.report)
print("type(r.report):", type(r.report))

try:
    print(r.report(99))
except TypeError as ex:
    print("Expected failure:", type(ex).__name__, ex)


r.report: <function report at 0x000002C81C7811C0>
type(r.report): <class 'function'>
Expected failure: TypeError report() missing 1 required positional argument: 'value'


## Solution 7A — Call the unbound instance-stored function explicitly

Because `report` lives in the instance dictionary, ordinary function descriptor binding through the class does not occur.


In [15]:
print("Instance dictionary:", r.__dict__)

# The function must receive the instance explicitly:
print(r.report(r, 99))

assert r.report(r, 99) == "R2: 99"


Instance dictionary: {'name': 'R2', 'report': <function report at 0x000002C81C7811C0>}
R2: 99


## Solution 7B — Bind it manually with `types.MethodType`


In [16]:
import types

r.report = types.MethodType(report, r)

print("r.report:", r.report)
print("type(r.report):", type(r.report))
print("r.report.__self__ is r:", r.report.__self__ is r)
print("r.report.__func__ is report:", r.report.__func__ is report)
print("r.report(99):", r.report(99))

assert r.report(99) == "R2: 99"


r.report: <bound method report of <__main__.Robot object at 0x000002C81C6DECF0>>
type(r.report): <class 'method'>
r.report.__self__ is r: True
r.report.__func__ is report: True
r.report(99): R2: 99


# Problem 8 — Manual descriptor binding with `__get__`

Python functions implement descriptor behavior.

Use:

```python
function.__get__(instance, owner_class)
```

to create a bound method manually.

Compare the result with normal attribute lookup.


In [17]:
class Multiplier:
    def __init__(self, factor):
        self.factor = factor

    def multiply(self, value):
        return self.factor * value

m = Multiplier(7)

function = Multiplier.__dict__["multiply"]

manual_bound = function.__get__(m, Multiplier)
normal_bound = m.multiply

print("function:", function)
print("manual_bound:", manual_bound)
print("normal_bound:", normal_bound)
print("manual_bound(6):", manual_bound(6))
print("normal_bound(6):", normal_bound(6))


function: <function Multiplier.multiply at 0x000002C81C781580>
manual_bound: <bound method Multiplier.multiply of <__main__.Multiplier object at 0x000002C81C6DF230>>
normal_bound: <bound method Multiplier.multiply of <__main__.Multiplier object at 0x000002C81C6DF230>>
manual_bound(6): 42
normal_bound(6): 42


## Solution 8


In [18]:
assert manual_bound.__func__ is function
assert manual_bound.__self__ is m

assert normal_bound.__func__ is function
assert normal_bound.__self__ is m

assert manual_bound(6) == normal_bound(6) == 42

print("Both bound methods wrap the same function and same instance.")


Both bound methods wrap the same function and same instance.


# Problem 9 — Observe `__get__` through the class

For a normal function stored in a class, investigate these two operations:

```python
function.__get__(None, Class)
function.__get__(instance, Class)
```

What is returned in each case?


In [19]:
class Sample:
    def action(self, x):
        return x * 2

fn = Sample.__dict__["action"]
obj = Sample()

through_class = fn.__get__(None, Sample)
through_instance = fn.__get__(obj, Sample)

print("fn:", fn)
print("fn.__get__(None, Sample):", through_class)
print("fn.__get__(obj, Sample):", through_instance)

print("type through class:", type(through_class))
print("type through instance:", type(through_instance))


fn: <function Sample.action at 0x000002C81C7816C0>
fn.__get__(None, Sample): <function Sample.action at 0x000002C81C7816C0>
fn.__get__(obj, Sample): <bound method Sample.action of <__main__.Sample object at 0x000002C81C6DF380>>
type through class: <class 'function'>
type through instance: <class 'method'>


## Solution 9


In [20]:
assert through_class is fn
assert through_instance.__func__ is fn
assert through_instance.__self__ is obj
assert through_instance(10) == 20

print("Class access leaves the function unbound.")
print("Instance access produces a bound method.")


Class access leaves the function unbound.
Instance access produces a bound method.


# Problem 10 — Method shadowing by an instance attribute

An instance can have an attribute with the same name as a method.

Predict what happens after assigning:

```python
x.compute = "disabled"
```

Can `Engine.compute(x, 5)` still be called?


In [21]:
class Engine:
    def compute(self, x):
        return x ** 2

engine = Engine()

print("Before shadowing:", engine.compute(5))

engine.compute = "disabled"

print("Instance dictionary:", engine.__dict__)
print("engine.compute:", engine.compute)

try:
    engine.compute(5)
except TypeError as ex:
    print("Expected:", type(ex).__name__, ex)

print("Class-level explicit call:", Engine.compute(engine, 5))


Before shadowing: 25
Instance dictionary: {'compute': 'disabled'}
engine.compute: disabled
Expected: TypeError 'str' object is not callable
Class-level explicit call: 25


## Solution 10

For an ordinary method function (a non-data descriptor), an instance attribute with the same name can shadow class lookup.

The class function is still available through the class:

```python
Engine.compute(engine, 5)
```


In [22]:
assert engine.compute == "disabled"
assert Engine.compute(engine, 5) == 25

del engine.compute

assert engine.compute(5) == 25
print("Deleting the shadowing instance attribute restores normal method lookup.")


Deleting the shadowing instance attribute restores normal method lookup.


# Problem 11 — Bound methods as callbacks

Store a bound method in a list of callbacks.

Then delete the original variable that referenced the instance.

Investigate whether the callback can still call the instance method.


In [23]:
class Logger:
    def __init__(self, prefix):
        self.prefix = prefix

    def format(self, message):
        return f"[{self.prefix}] {message}"

logger = Logger("APP")
callback = logger.format

callbacks = [callback]

print(callbacks[0]("started"))

print("callback.__self__:", callback.__self__)
print("callback.__func__:", callback.__func__)


[APP] started
callback.__self__: <__main__.Logger object at 0x000002C81C6DF4D0>
callback.__func__: <function Logger.format at 0x000002C81C7814E0>


## Solution 11

A bound method holds a reference to its bound object in `.__self__`.

Therefore, keeping the bound method alive also keeps a reference to the instance.


In [24]:
import weakref
import gc

logger2 = Logger("TEMP")
ref = weakref.ref(logger2)
callback2 = logger2.format

print("Alive before del:", ref() is not None)

del logger2
gc.collect()

print("Alive while bound method exists:", ref() is not None)
print("Callback still works:", callback2("message"))

del callback2
gc.collect()

print("Alive after bound method reference removed:", ref() is not None)


Alive before del: True
Alive while bound method exists: True
Callback still works: [TEMP] message
Alive after bound method reference removed: False


# Problem 12 — Bound method from one instance, called later

Create two instances of the same class.

Store a bound method from the first instance, then reassign the variable that originally pointed to that first instance.

Which object does the stored method use?


In [25]:
class Box:
    def __init__(self, value):
        self.value = value

    def read(self):
        return self.value

a = Box("A")
b = Box("B")

reader = a.read

a = b

print("a.read():", a.read())
print("reader():", reader())
print("reader.__self__.value:", reader.__self__.value)


a.read(): B
reader(): A
reader.__self__.value: A


## Solution 12

The stored bound method keeps its original binding.

Reassigning the *variable* `a` does not mutate the method's `.__self__`.


In [26]:
assert a.read() == "B"
assert reader() == "A"
assert reader.__self__.value == "A"


# Problem 13 — A subtle loop callback bug

Build callbacks for several instances.

First, demonstrate a late-binding closure mistake.

Then solve it by storing bound methods directly.


In [27]:
class Task:
    def __init__(self, name):
        self.name = name

    def run(self):
        return f"running {self.name}"

tasks = [Task("A"), Task("B"), Task("C")]

# Buggy closure approach:
buggy = []
for task in tasks:
    buggy.append(lambda: task.run())

print([fn() for fn in buggy])


['running C', 'running C', 'running C']


## Solution 13A — Explain the bug

The lambdas close over the loop variable `task`; they do not snapshot its value on each iteration. After the loop, `task` refers to the final object.


In [28]:
print("Final loop variable:", task.name)
assert [fn() for fn in buggy] == ["running C", "running C", "running C"]


Final loop variable: C


## Solution 13B — Store bound methods directly


In [29]:
callbacks = [task.run for task in tasks]

results = [fn() for fn in callbacks]

print(results)

assert results == ["running A", "running B", "running C"]

for i, cb in enumerate(callbacks):
    print(i, cb.__self__.name, cb.__func__.__name__)


['running A', 'running B', 'running C']
0 A run
1 B run
2 C run


# Problem 14 — Calling an instance method through the wrong instance type

Python does not enforce the nominal class of `self` when a plain function is called explicitly through the class.

Explore what happens when the supplied object merely has the required attributes.


In [30]:
class Formatter:
    def render(self):
        return f"<{self.value}>"

class Compatible:
    def __init__(self):
        self.value = "works"

class Incompatible:
    pass

compatible = Compatible()
incompatible = Incompatible()

print(Formatter.render(compatible))

try:
    print(Formatter.render(incompatible))
except AttributeError as ex:
    print("Expected:", type(ex).__name__, ex)


<works>
Expected: AttributeError 'Incompatible' object has no attribute 'value'


## Solution 14

When calling `Formatter.render(obj)` directly, the function receives `obj` as its first argument. The function body determines what attributes are required.

This is one reason Python's object model can support duck-typed behavior.


In [31]:
assert Formatter.render(compatible) == "<works>"


# Problem 15 — Inheritance and binding

Predict the values of `.__func__` and `.__self__` for a method inherited from a base class.


In [32]:
class Base:
    def identify(self):
        return f"instance type = {type(self).__name__}"

class Child(Base):
    pass

child = Child()
method = child.identify

print(method)
print("method.__func__:", method.__func__)
print("method.__self__:", method.__self__)
print("method():", method())


<bound method Base.identify of <__main__.Child object at 0x000002C81C7B0830>>
method.__func__: <function Base.identify at 0x000002C81C7818A0>
method.__self__: <__main__.Child object at 0x000002C81C7B0830>
method(): instance type = Child


## Solution 15


In [33]:
assert method.__func__ is Base.identify
assert method.__self__ is child
assert method() == "instance type = Child"

print("The inherited function is Base.identify, but it binds to the Child instance.")


The inherited function is Base.identify, but it binds to the Child instance.


# Problem 16 — Override vs inherited function

Now override the method in the child class.

Compare:

- `Base.identify`
- `Child.identify`
- `child.identify.__func__`


In [34]:
class Base:
    def identify(self):
        return "Base implementation"

class Child(Base):
    def identify(self):
        return "Child implementation"

child = Child()

print("Base.identify:", Base.identify)
print("Child.identify:", Child.identify)
print("child.identify.__func__:", child.identify.__func__)
print("child.identify():", child.identify())


Base.identify: <function Base.identify at 0x000002C81C781C60>
Child.identify: <function Child.identify at 0x000002C81C781E40>
child.identify.__func__: <function Child.identify at 0x000002C81C781E40>
child.identify(): Child implementation


## Solution 16


In [35]:
assert Base.identify is not Child.identify
assert child.identify.__func__ is Child.identify
assert child.identify.__self__ is child
assert child.identify() == "Child implementation"


# Problem 17 — Capture a base-class implementation explicitly

Even when a subclass overrides a method, you can access the base implementation explicitly.

Compare:

```python
Base.identify(child)
Base.identify.__get__(child, Child)()
```


In [36]:
print(Base.identify(child))

base_bound = Base.identify.__get__(child, Child)

print(base_bound)
print(base_bound())


Base implementation
<bound method Base.identify of <__main__.Child object at 0x000002C81C7B0440>>
Base implementation


## Solution 17


In [37]:
assert base_bound.__func__ is Base.identify
assert base_bound.__self__ is child
assert base_bound() == "Base implementation"
assert Base.identify(child) == "Base implementation"


# Problem 18 — Does copying a bound method copy the instance?

Use `copy.copy` on a bound method and inspect the result.

The goal is not to memorize implementation trivia; the goal is to identify what binding information must remain consistent.


In [38]:
import copy

class State:
    def __init__(self, value):
        self.value = value

    def get(self):
        return self.value

s = State(123)
m1 = s.get
m2 = copy.copy(m1)

print("m1:", m1)
print("m2:", m2)
print("same __self__:", m1.__self__ is m2.__self__)
print("same __func__:", m1.__func__ is m2.__func__)
print("m2():", m2())


m1: <bound method State.get of <__main__.State object at 0x000002C81C7B0590>>
m2: <bound method State.get of <__main__.State object at 0x000002C81C7B0590>>
same __self__: True
same __func__: True
m2(): 123


## Solution 18

The important semantic relationship is that both callable objects refer to the same underlying function and bound instance.


In [39]:
assert m1.__self__ is m2.__self__
assert m1.__func__ is m2.__func__
assert m2() == 123


# Problem 19 — Build a method-inspection utility

Write:

```python
inspect_callable(obj)
```

that returns a dictionary containing as much of the following information as is available:

- runtime type name
- `callable(obj)`
- `__name__`
- `__qualname__`
- whether it has `__func__`
- whether it has `__self__`
- underlying function
- bound object

Test it on:

- a plain function
- a class function accessed through the class
- a bound method
- a lambda stored directly on an instance


In [40]:
def inspect_callable(obj):
    # TODO: implement
    pass


## Solution 19


In [41]:
def inspect_callable(obj):
    return {
        "type": type(obj).__name__,
        "callable": callable(obj),
        "name": getattr(obj, "__name__", None),
        "qualname": getattr(obj, "__qualname__", None),
        "has___func__": hasattr(obj, "__func__"),
        "has___self__": hasattr(obj, "__self__"),
        "func": getattr(obj, "__func__", None),
        "self": getattr(obj, "__self__", None),
    }

def plain(x):
    return x

class Demo:
    def method(self, x):
        return x

demo = Demo()
demo.direct = lambda x: x

objects = {
    "plain": plain,
    "Demo.method": Demo.method,
    "demo.method": demo.method,
    "demo.direct": demo.direct,
}

for label, obj in objects.items():
    print("\n", label)
    for key, value in inspect_callable(obj).items():
        print(f"  {key}: {value}")



 plain
  type: function
  callable: True
  name: plain
  qualname: plain
  has___func__: False
  has___self__: False
  func: None
  self: None

 Demo.method
  type: function
  callable: True
  name: method
  qualname: Demo.method
  has___func__: False
  has___self__: False
  func: None
  self: None

 demo.method
  type: method
  callable: True
  name: method
  qualname: Demo.method
  has___func__: True
  has___self__: True
  func: <function Demo.method at 0x000002C81C782340>
  self: <__main__.Demo object at 0x000002C81C7B0980>

 demo.direct
  type: function
  callable: True
  name: <lambda>
  qualname: <lambda>
  has___func__: False
  has___self__: False
  func: None
  self: None


# Problem 20 — Reconstruct method invocation generically

Write a function:

```python
invoke_bound_method(method, *args, **kwargs)
```

that does **not** call `method(...)` directly.

Instead, it must use:

```python
method.__func__
method.__self__
```

to reproduce the call.


In [42]:
def invoke_bound_method(method, *args, **kwargs):
    # TODO
    pass


## Solution 20


In [43]:
def invoke_bound_method(method, *args, **kwargs):
    if not hasattr(method, "__func__") or not hasattr(method, "__self__"):
        raise TypeError("Expected a bound method")
    return method.__func__(method.__self__, *args, **kwargs)

class Calculator:
    def combine(self, a, b=0):
        return a + b

calc = Calculator()

print(invoke_bound_method(calc.combine, 10, b=5))

assert invoke_bound_method(calc.combine, 10, b=5) == 15


15


# Problem 21 — Detect whether two bound methods represent the same binding

Implement:

```python
same_binding(a, b)
```

Return `True` only when both objects expose the same:

- `__func__`
- `__self__`

Use identity checks.


In [44]:
def same_binding(a, b):
    # TODO
    pass


## Solution 21


In [45]:
def same_binding(a, b):
    return (
        hasattr(a, "__func__")
        and hasattr(a, "__self__")
        and hasattr(b, "__func__")
        and hasattr(b, "__self__")
        and a.__func__ is b.__func__
        and a.__self__ is b.__self__
    )

class A:
    def f(self):
        return 1

a1 = A()
a2 = A()

print(same_binding(a1.f, a1.f))
print(same_binding(a1.f, a2.f))

assert same_binding(a1.f, a1.f)
assert not same_binding(a1.f, a2.f)


True
False


# Problem 22 — Build a per-instance method override

Override a method for **one instance only** without modifying the class or other instances.

Requirements:

- `a.speak()` should use the override.
- `b.speak()` should keep the original class behavior.
- `Speaker.speak` should remain unchanged.

Use `types.MethodType`.


In [46]:
import types

class Speaker:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return f"{self.name}: default"

a = Speaker("A")
b = Speaker("B")

original_function = Speaker.speak

def custom_speak(self):
    return f"{self.name}: custom"

# TODO: override only a.speak


## Solution 22


In [47]:
a.speak = types.MethodType(custom_speak, a)

print(a.speak())
print(b.speak())
print(Speaker.speak is original_function)

assert a.speak() == "A: custom"
assert b.speak() == "B: default"
assert Speaker.speak is original_function
assert a.speak.__self__ is a
assert a.speak.__func__ is custom_speak


A: custom
B: default
True


# Problem 23 — Replace a class method at runtime and observe old bound references

This is an important monkey-patching edge case.

Steps:

1. obtain a bound method from an instance
2. replace the class function with a new function
3. call the *old stored bound method*
4. call a *new lookup* from the same instance

Predict whether the old stored bound method changes.


In [48]:
class Service:
    def operation(self):
        return "version 1"

service = Service()

old_bound = service.operation

def operation_v2(self):
    return "version 2"

Service.operation = operation_v2

print("Old stored method:", old_bound())
print("New lookup:", service.operation())

print("old_bound.__func__:", old_bound.__func__)
print("service.operation.__func__:", service.operation.__func__)


Old stored method: version 1
New lookup: version 2
old_bound.__func__: <function Service.operation at 0x000002C81C782480>
service.operation.__func__: <function operation_v2 at 0x000002C81C782840>


## Solution 23

The old bound method keeps the function it captured when that bound-method object was created.

A later attribute lookup sees the class's newly assigned function and creates a bound method around the new function.


In [49]:
assert old_bound() == "version 1"
assert service.operation() == "version 2"
assert old_bound.__func__ is not service.operation.__func__
assert service.operation.__func__ is operation_v2


# Problem 24 — Function attributes survive binding through `__func__`

Functions themselves can have custom attributes.

Attach metadata to a class function, retrieve a bound method, and access the metadata through `.__func__`.


In [50]:
class Endpoint:
    def handle(self, request):
        return f"handling {request}"

Endpoint.handle.route = "/items"
Endpoint.handle.requires_auth = True

endpoint = Endpoint()
bound = endpoint.handle

print(bound.__func__.route)
print(bound.__func__.requires_auth)


/items
True


## Solution 24


In [51]:
assert bound.__func__ is Endpoint.handle
assert bound.__func__.route == "/items"
assert bound.__func__.requires_auth is True

print("Function metadata remains attached to the underlying function object.")


Function metadata remains attached to the underlying function object.


# Problem 25 — Decorator that preserves method binding

Write a decorator that logs calls but still behaves correctly as an instance method.

Use `functools.wraps`.


In [52]:
from functools import wraps

def trace(func):
    # TODO
    pass


## Solution 25


In [53]:
from functools import wraps

def trace(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"TRACE {func.__qualname__} args={args!r} kwargs={kwargs!r}")
        return func(*args, **kwargs)
    return wrapper

class Store:
    def __init__(self, name):
        self.name = name

    @trace
    def fetch(self, key):
        return f"{self.name}:{key}"

store = Store("cache")

print(store.fetch("x"))
print(store.fetch.__self__ is store)
print(store.fetch.__func__.__name__)

assert store.fetch("x") == "cache:x"
assert store.fetch.__self__ is store
assert store.fetch.__func__.__name__ == "fetch"


TRACE Store.fetch args=(<__main__.Store object at 0x000002C81C7B1160>, 'x') kwargs={}
cache:x
True
fetch
TRACE Store.fetch args=(<__main__.Store object at 0x000002C81C7B1160>, 'x') kwargs={}


# Problem 26 — Decorator bug: forgetting the instance argument

Diagnose the failure.


In [54]:
def bad_decorator(func):
    def wrapper():
        return func()
    return wrapper

class Example:
    @bad_decorator
    def action(self):
        return "ok"

e = Example()

try:
    print(e.action())
except TypeError as ex:
    print("Expected:", type(ex).__name__, ex)


Expected: TypeError bad_decorator.<locals>.wrapper() takes 0 positional arguments but 1 was given


## Solution 26

The decorated function stored in the class is now `wrapper`.

When `e.action` is accessed, `wrapper` is bound to `e`, so `e` is supplied to `wrapper`. But `wrapper` declares no parameters.

A robust general-purpose method decorator normally accepts `*args, **kwargs` and forwards them.


In [55]:
def good_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

class Fixed:
    @good_decorator
    def action(self):
        return "ok"

fixed = Fixed()

assert fixed.action() == "ok"
print(fixed.action())


ok


# Problem 27 — Build a tiny event system with bound callbacks

Implement an `Event` class with:

- `subscribe(callback)`
- `emit(*args, **kwargs)`

Subscribe bound methods from multiple instances and verify that each callback retains its own instance binding.


In [56]:
class Event:
    def __init__(self):
        self._callbacks = []

    def subscribe(self, callback):
        # TODO
        pass

    def emit(self, *args, **kwargs):
        # TODO
        pass


## Solution 27


In [57]:
class Event:
    def __init__(self):
        self._callbacks = []

    def subscribe(self, callback):
        if not callable(callback):
            raise TypeError("callback must be callable")
        self._callbacks.append(callback)

    def emit(self, *args, **kwargs):
        return [callback(*args, **kwargs) for callback in self._callbacks]

class Listener:
    def __init__(self, name):
        self.name = name

    def on_message(self, message):
        return f"{self.name} received {message}"

event = Event()

l1 = Listener("L1")
l2 = Listener("L2")

event.subscribe(l1.on_message)
event.subscribe(l2.on_message)

results = event.emit("hello")

print(results)

assert results == [
    "L1 received hello",
    "L2 received hello",
]

for cb in event._callbacks:
    print(cb.__self__.name, cb.__func__.__name__)


['L1 received hello', 'L2 received hello']
L1 on_message
L2 on_message


# Problem 28 — Remove a bound callback reliably

Because repeated method access can create fresh bound-method objects, write an `unsubscribe` method that removes a callback by comparing binding structure instead of relying only on `is`.

Use the `same_binding` helper from Problem 21.


In [58]:
class Event:
    def __init__(self):
        self._callbacks = []

    def subscribe(self, callback):
        self._callbacks.append(callback)

    def unsubscribe(self, callback):
        # TODO
        pass

    def emit(self, *args, **kwargs):
        return [cb(*args, **kwargs) for cb in self._callbacks]


## Solution 28


In [59]:
class Event:
    def __init__(self):
        self._callbacks = []

    def subscribe(self, callback):
        self._callbacks.append(callback)

    def unsubscribe(self, callback):
        new_callbacks = []
        removed = False

        for existing in self._callbacks:
            if same_binding(existing, callback):
                removed = True
            else:
                new_callbacks.append(existing)

        self._callbacks = new_callbacks
        return removed

    def emit(self, *args, **kwargs):
        return [cb(*args, **kwargs) for cb in self._callbacks]

listener = Listener("solo")
event = Event()

event.subscribe(listener.on_message)

print(event.emit("before"))
print("removed:", event.unsubscribe(listener.on_message))
print(event.emit("after"))

assert event.emit("after") == []


['solo received before']
removed: True
[]


# Problem 29 — Inspect class namespace vs instance namespace

Create a class with:

- one class attribute
- one instance attribute
- one instance method

Then print:

- `Class.__dict__`
- `instance.__dict__`

Explain why the method is absent from `instance.__dict__` even though `instance.method` works.


In [60]:
class Record:
    category = "demo"

    def __init__(self, value):
        self.value = value

    def show(self):
        return self.value

record = Record(42)

print("Record keys:")
print(sorted(k for k in Record.__dict__ if not k.startswith("__")))

print("\nrecord.__dict__:")
print(record.__dict__)

print("\nrecord.show:")
print(record.show)


Record keys:
['category', 'show']

record.__dict__:
{'value': 42}

record.show:
<bound method Record.show of <__main__.Record object at 0x000002C81C7B1550>>


## Solution 29

`show` is stored in the class namespace.

Attribute lookup on the instance can find class attributes. Because the class attribute is a function descriptor, instance lookup converts it into a bound method.

The bound method is normally produced during lookup; it does not need to be stored in `record.__dict__`.


In [61]:
assert "show" in Record.__dict__
assert "show" not in record.__dict__
assert record.show.__func__ is Record.__dict__["show"]
assert record.show.__self__ is record


# Problem 30 — Implement a descriptor that mimics basic method binding

This advanced extension builds a tiny descriptor that wraps a function.

Requirements:

- class access returns the underlying function
- instance access returns a callable bound to the instance
- use `types.MethodType`


In [62]:
import types

class MethodLike:
    def __init__(self, func):
        self.func = func

    def __get__(self, instance, owner):
        # TODO
        pass


## Solution 30


In [63]:
import types

class MethodLike:
    def __init__(self, func):
        self.func = func

    def __get__(self, instance, owner):
        if instance is None:
            return self.func
        return types.MethodType(self.func, instance)

class DemoDescriptor:
    def _show(self, x):
        return f"{self.name}:{x}"

    show = MethodLike(_show)

    def __init__(self, name):
        self.name = name

dd = DemoDescriptor("descriptor")

print("Class access:", DemoDescriptor.show)
print("Instance access:", dd.show)
print("Call:", dd.show(10))

assert DemoDescriptor.show is DemoDescriptor._show
assert dd.show.__self__ is dd
assert dd.show.__func__ is DemoDescriptor._show
assert dd.show(10) == "descriptor:10"


Class access: <function DemoDescriptor._show at 0x000002C81C783EC0>
Instance access: <bound method DemoDescriptor._show of <__main__.DemoDescriptor object at 0x000002C81C7B1A90>>
Call: descriptor:10


# Problem 31 — Implement a minimal custom bound-method object

Instead of using `types.MethodType`, create your own callable object that stores:

- a function
- an instance

Its `__call__` should invoke:

```python
function(instance, *args, **kwargs)
```

Then use it from a descriptor.


In [64]:
class MiniBoundMethod:
    def __init__(self, func, instance):
        # TODO
        pass

    def __call__(self, *args, **kwargs):
        # TODO
        pass

class MiniMethod:
    def __init__(self, func):
        self.func = func

    def __get__(self, instance, owner):
        # TODO
        pass


## Solution 31


In [65]:
class MiniBoundMethod:
    def __init__(self, func, instance):
        self.__func__ = func
        self.__self__ = instance

    def __call__(self, *args, **kwargs):
        return self.__func__(self.__self__, *args, **kwargs)

    def __repr__(self):
        return (
            f"<MiniBoundMethod "
            f"{self.__func__.__qualname__} "
            f"of {self.__self__!r}>"
        )

class MiniMethod:
    def __init__(self, func):
        self.func = func

    def __get__(self, instance, owner):
        if instance is None:
            return self.func
        return MiniBoundMethod(self.func, instance)

class MiniExample:
    def _scale(self, value):
        return self.factor * value

    scale = MiniMethod(_scale)

    def __init__(self, factor):
        self.factor = factor

mini = MiniExample(8)

print(mini.scale)
print(mini.scale(5))
print(mini.scale.__self__)
print(mini.scale.__func__)

assert mini.scale(5) == 40


<MiniBoundMethod MiniExample._scale of <__main__.MiniExample object at 0x000002C81C7B1D30>>
40
<function MiniExample._scale at 0x000002C81C7EC4A0>


# Problem 32 — Why class-level and instance-level assignment differ

Compare these two operations:

```python
Class.f = function
instance.g = function
```

Create a compact experiment that proves:

- `instance.f` becomes a bound method
- `instance.g` remains a plain function


In [66]:
class Target:
    pass

def external(self, x):
    return self, x

Target.f = external

t = Target()
t.g = external

print("Target.f:", Target.f)
print("t.f:", t.f)
print("t.g:", t.g)

print("type(t.f):", type(t.f))
print("type(t.g):", type(t.g))


Target.f: <function external at 0x000002C81C783C40>
t.f: <bound method external of <__main__.Target object at 0x000002C81C7B1E80>>
t.g: <function external at 0x000002C81C783C40>
type(t.f): <class 'method'>
type(t.g): <class 'function'>


## Solution 32


In [67]:
assert hasattr(t.f, "__self__")
assert t.f.__self__ is t
assert t.f.__func__ is external

assert not hasattr(t.g, "__self__")
assert t.g is external

bound_self, value = t.f(1)
plain_self, value2 = t.g(t, 2)

assert bound_self is t and value == 1
assert plain_self is t and value2 == 2


# Problem 33 — Method objects as first-class values

Create a processing pipeline from bound methods.

Each object should transform a value differently.


In [68]:
class Add:
    def __init__(self, amount):
        self.amount = amount

    def apply(self, x):
        return x + self.amount

class Multiply:
    def __init__(self, factor):
        self.factor = factor

    def apply(self, x):
        return x * self.factor

steps = [
    Add(3).apply,
    Multiply(10).apply,
    Add(-5).apply,
]

value = 2

for step in steps:
    value = step(value)
    print(value)


5
50
45


## Solution 33


In [69]:
# 2 -> 5 -> 50 -> 45
assert value == 45

for step in steps:
    print(
        "function =", step.__func__.__qualname__,
        "| bound object =", step.__self__,
    )


function = Add.apply | bound object = <__main__.Add object at 0x000002C81C7B2120>
function = Multiply.apply | bound object = <__main__.Multiply object at 0x000002C81C7B2510>
function = Add.apply | bound object = <__main__.Add object at 0x000002C81C775A90>


# Problem 34 — Challenge: method cache

Build a cache keyed by:

```python
(instance, underlying_function)
```

Given repeated bound-method lookups, normalize them to the same logical key.


In [70]:
def method_key(method):
    # TODO
    pass


## Solution 34


In [71]:
def method_key(method):
    if not hasattr(method, "__func__") or not hasattr(method, "__self__"):
        raise TypeError("Expected bound method")
    return (method.__self__, method.__func__)

class CacheDemo:
    def work(self):
        return "work"

obj = CacheDemo()

k1 = method_key(obj.work)
k2 = method_key(obj.work)

print(k1)
print(k2)
print("same logical key:", k1 == k2)

assert k1 == k2
assert k1[0] is obj
assert k1[1] is CacheDemo.work


(<__main__.CacheDemo object at 0x000002C81C7B27B0>, <function CacheDemo.work at 0x000002C81C7ECAE0>)
(<__main__.CacheDemo object at 0x000002C81C7B27B0>, <function CacheDemo.work at 0x000002C81C7ECAE0>)
same logical key: True


# Problem 35 — Challenge: generic rebinding

Write:

```python
rebind(method, new_instance)
```

that takes a bound method and returns a new bound method using the *same underlying function* but a different instance.

Use `types.MethodType`.


In [72]:
import types

def rebind(method, new_instance):
    # TODO
    pass


## Solution 35


In [73]:
import types

def rebind(method, new_instance):
    if not hasattr(method, "__func__"):
        raise TypeError("Expected bound method")
    return types.MethodType(method.__func__, new_instance)

class Label:
    def __init__(self, text):
        self.text = text

    def show(self):
        return self.text

one = Label("one")
two = Label("two")

m1 = one.show
m2 = rebind(m1, two)

print(m1())
print(m2())

assert m1.__func__ is m2.__func__ is Label.show
assert m1.__self__ is one
assert m2.__self__ is two
assert m2() == "two"


one
two


# Problem 36 — Challenge: intentionally bypass an instance override

Suppose one instance shadows a class method.

Write a helper that calls the original class implementation directly.


In [74]:
class Processor:
    def process(self, x):
        return x * 2

p = Processor()

p.process = lambda x: -1

print("Instance override:", p.process(10))

def call_class_implementation(instance, method_name, *args, **kwargs):
    # TODO
    pass


Instance override: -1


## Solution 36


In [75]:
def call_class_implementation(instance, method_name, *args, **kwargs):
    cls = type(instance)
    function = getattr(cls, method_name)
    return function(instance, *args, **kwargs)

print("Class implementation:", call_class_implementation(p, "process", 10))

assert p.process(10) == -1
assert call_class_implementation(p, "process", 10) == 20


Class implementation: 20


# Problem 37 — Debugging exercise: accidental replacement of a method

Find the bug and repair the object without constructing a new instance.


In [76]:
class Session:
    def status(self):
        return "active"

s = Session()

print("Initially:", s.status())

# Bug:
s.status = "closed"

print("After accidental overwrite:", s.status)

try:
    s.status()
except TypeError as ex:
    print("Failure:", type(ex).__name__, ex)


Initially: active
After accidental overwrite: closed
Failure: TypeError 'str' object is not callable


## Solution 37

The class method still exists. The instance attribute is shadowing it.

Delete the shadowing instance attribute.


In [77]:
del s.status

print(s.status())

assert s.status() == "active"
assert "status" not in s.__dict__
assert "status" in Session.__dict__


active


# Problem 38 — Debugging exercise: callback lost `self`

A programmer stores `Handler.handle` instead of `handler.handle`.

Explain the difference and fix the callback registration.


In [78]:
class Handler:
    def __init__(self, name):
        self.name = name

    def handle(self, value):
        return f"{self.name}:{value}"

handler = Handler("H")

bad_callback = Handler.handle
good_callback = handler.handle

try:
    print(bad_callback(10))
except TypeError as ex:
    print("Bad callback:", type(ex).__name__, ex)

print("Good callback:", good_callback(10))


Bad callback: TypeError Handler.handle() missing 1 required positional argument: 'value'
Good callback: H:10


## Solution 38

`Handler.handle` is the unbound function accessed through the class. It still expects an instance as its first argument.

`handler.handle` is a bound method and already remembers `handler`.


In [79]:
assert bad_callback(handler, 10) == "H:10"
assert good_callback(10) == "H:10"
assert good_callback.__self__ is handler


# Problem 39 — Debugging exercise: wrong instance rebound

The same underlying function can be bound to multiple compatible instances.

Track which object receives each call.


In [80]:
class Named:
    def __init__(self, name):
        self.name = name

    def who(self):
        return self.name

x = Named("X")
y = Named("Y")

mx = x.who
my = types.MethodType(mx.__func__, y)

print(mx())
print(my())

assert mx.__self__ is x
assert my.__self__ is y
assert mx.__func__ is my.__func__


X
Y


# Problem 40 — Capstone: dynamic command registry

Build a command registry with these features:

1. register bound instance methods
2. execute commands by name
3. inspect which instance and underlying function each command uses
4. replace one command with a rebound version using another compatible instance
5. keep the registry independent of the original local variable names

This capstone combines first-class bound methods, `__self__`, `__func__`, and rebinding.


In [81]:
class CommandRegistry:
    def __init__(self):
        self._commands = {}

    def register(self, name, method):
        # TODO
        pass

    def execute(self, name, *args, **kwargs):
        # TODO
        pass

    def describe(self, name):
        # TODO
        pass

    def rebind(self, name, new_instance):
        # TODO
        pass


## Solution 40


In [82]:
class CommandRegistry:
    def __init__(self):
        self._commands = {}

    def register(self, name, method):
        if not (
            hasattr(method, "__func__")
            and hasattr(method, "__self__")
            and method.__self__ is not None
        ):
            raise TypeError("register expects a bound instance method")

        self._commands[name] = method

    def execute(self, name, *args, **kwargs):
        return self._commands[name](*args, **kwargs)

    def describe(self, name):
        method = self._commands[name]
        return {
            "name": name,
            "function": method.__func__.__qualname__,
            "instance_type": type(method.__self__).__name__,
            "instance": method.__self__,
        }

    def rebind(self, name, new_instance):
        old_method = self._commands[name]
        self._commands[name] = types.MethodType(
            old_method.__func__,
            new_instance,
        )

class CalculatorService:
    def __init__(self, bias):
        self.bias = bias

    def calculate(self, x):
        return x + self.bias

service_a = CalculatorService(100)
service_b = CalculatorService(1000)

registry = CommandRegistry()

registry.register("calc", service_a.calculate)

print("Original:", registry.execute("calc", 5))
print("Description:", registry.describe("calc"))

registry.rebind("calc", service_b)

print("Rebound:", registry.execute("calc", 5))
print("Description:", registry.describe("calc"))

assert registry.execute("calc", 5) == 1005
assert registry._commands["calc"].__self__ is service_b
assert registry._commands["calc"].__func__ is CalculatorService.calculate


Original: 105
Description: {'name': 'calc', 'function': 'CalculatorService.calculate', 'instance_type': 'CalculatorService', 'instance': <__main__.CalculatorService object at 0x000002C81C7B2CF0>}
Rebound: 1005
Description: {'name': 'calc', 'function': 'CalculatorService.calculate', 'instance_type': 'CalculatorService', 'instance': <__main__.CalculatorService object at 0x000002C81C775D10>}


# Extra Example A — Bound method unpacking helper


In [83]:
def split_bound_method(method):
    if not hasattr(method, "__func__") or not hasattr(method, "__self__"):
        raise TypeError("Not a bound method")
    return method.__func__, method.__self__

class ExampleA:
    def f(self, x):
        return x + 1

ea = ExampleA()
func, instance = split_bound_method(ea.f)

print(func)
print(instance)
print(func(instance, 10))

assert func(instance, 10) == ea.f(10) == 11


<function ExampleA.f at 0x000002C81C7EC180>
11


# Extra Example B — Multiple bound methods share the same instance


In [84]:
class Multi:
    def first(self):
        return "first"

    def second(self):
        return "second"

multi = Multi()

first = multi.first
second = multi.second

print(first.__self__ is second.__self__)
print(first.__func__ is second.__func__)

assert first.__self__ is second.__self__ is multi
assert first.__func__ is Multi.first
assert second.__func__ is Multi.second


True
False


# Extra Example C — Same function bound to different instances


In [85]:
class Value:
    def __init__(self, value):
        self.value = value

    def get(self):
        return self.value

v1 = Value(1)
v2 = Value(2)

m1 = v1.get
m2 = v2.get

print(m1())
print(m2())

assert m1.__func__ is m2.__func__ is Value.get
assert m1.__self__ is v1
assert m2.__self__ is v2


1
2


# Extra Example D — Store a bound method in another object


In [86]:
class Producer:
    def produce(self, x):
        return x * 3

class Holder:
    def __init__(self, callback):
        self.callback = callback

producer = Producer()
holder = Holder(producer.produce)

print(holder.callback(7))
print(holder.callback.__self__ is producer)

assert holder.callback(7) == 21


21
True


# Extra Example E — A function added to a class after instances already exist


In [87]:
class Existing:
    def __init__(self, value):
        self.value = value

old_instance = Existing(50)

def double(self):
    return self.value * 2

Existing.double = double

print(old_instance.double())

assert old_instance.double() == 100
assert old_instance.double.__func__ is double
assert old_instance.double.__self__ is old_instance


100


# Extra Example F — An instance-stored callable object

Not every callable stored on an instance is a method.


In [88]:
class CallableObject:
    def __call__(self, x):
        return x ** 3

class Container:
    pass

container = Container()
container.operation = CallableObject()

print(container.operation)
print(type(container.operation))
print(container.operation(4))

assert container.operation(4) == 64
assert not hasattr(container.operation, "__func__")


<class '__main__.CallableObject'>
64


# Extra Example G — Bound methods and `callable`


In [89]:
class C:
    def f(self):
        pass

c = C()

objects = [
    C.f,
    c.f,
    lambda: None,
    123,
    "hello",
]

for obj in objects:
    print(repr(obj), "-> callable:", callable(obj))


<function C.f at 0x000002C81C783740> -> callable: True
<bound method C.f of <__main__.C object at 0x000002C81C7B3E00>> -> callable: True
<function <lambda> at 0x000002C81C783600> -> callable: True
123 -> callable: False
'hello' -> callable: False


# Extra Example H — A concise equivalence tester

For a normal instance method, these two calls should agree:

```python
obj.method(*args)
type(obj).method(obj, *args)
```


In [90]:
class Power:
    def raise_to(self, base, exponent):
        return base ** exponent

power = Power()

via_instance = power.raise_to(2, 8)
via_class = Power.raise_to(power, 2, 8)

print(via_instance, via_class)

assert via_instance == via_class == 256


256 256


# Extra Example I — A method can mutate exactly the object it is bound to


In [91]:
class Bucket:
    def __init__(self):
        self.items = []

    def add(self, item):
        self.items.append(item)

left = Bucket()
right = Bucket()

add_left = left.add
add_right = right.add

add_left("A")
add_left("B")
add_right("X")

print("left:", left.items)
print("right:", right.items)

assert left.items == ["A", "B"]
assert right.items == ["X"]


left: ['A', 'B']
right: ['X']


# Extra Example J — Bound method metadata table


In [92]:
class MetaDemo:
    def alpha(self):
        return "alpha"

    def beta(self):
        return "beta"

obj = MetaDemo()

for method in [obj.alpha, obj.beta]:
    print({
        "method_repr": repr(method),
        "function_name": method.__func__.__name__,
        "qualname": method.__func__.__qualname__,
        "instance_id": id(method.__self__),
        "instance_type": type(method.__self__).__name__,
    })


{'method_repr': '<bound method MetaDemo.alpha of <__main__.MetaDemo object at 0x000002C81C81C050>>', 'function_name': 'alpha', 'qualname': 'MetaDemo.alpha', 'instance_id': 3058494980176, 'instance_type': 'MetaDemo'}
{'method_repr': '<bound method MetaDemo.beta of <__main__.MetaDemo object at 0x000002C81C81C050>>', 'function_name': 'beta', 'qualname': 'MetaDemo.beta', 'instance_id': 3058494980176, 'instance_type': 'MetaDemo'}


# Final Review — 20 rapid-fire questions

Try to answer these before looking at the answer key.

1. Where is a normally defined instance method function stored?
2. What does `obj.method` usually produce?
3. What does `Class.method` usually produce for a normal method?
4. What does `method.__self__` contain?
5. What does `method.__func__` contain?
6. Is `self` a required keyword?
7. Why does a zero-parameter class function fail when called through an instance?
8. Is `obj.method()` behaviorally similar to `Class.method(obj)`?
9. Does a plain function assigned directly to `obj.x` automatically become bound?
10. Does a function assigned to `Class.x` normally bind on instance lookup?
11. How can a function be manually bound to an instance?
12. What protocol makes normal function binding possible?
13. Can an instance attribute shadow an ordinary method?
14. Does a stored bound method remember its instance?
15. Can the same function be bound to different instances?
16. Can two different methods be bound to the same instance?
17. Does replacing a class function retroactively mutate already-stored bound methods?
18. Can an inherited base-class function bind to a subclass instance?
19. What two identity checks can test the structure of a binding?
20. Why are bound methods useful as callbacks?


## Final Review — Answer key

1. In the class namespace / class `__dict__`.
2. A bound method.
3. The function object.
4. The bound instance.
5. The underlying function.
6. No. It is a convention.
7. Instance access supplies the instance as the first positional argument.
8. Yes, for an ordinary instance method.
9. No.
10. Yes, when accessed through an instance.
11. For example with `types.MethodType(function, instance)` or `function.__get__(instance, owner)`.
12. The descriptor protocol, especially `__get__`.
13. Yes, for ordinary non-data-descriptor method functions.
14. Yes, through `.__self__`.
15. Yes.
16. Yes.
17. No. An already-created bound method retains its stored function.
18. Yes.
19. Compare `.__func__ is ...` and `.__self__ is ...`.
20. They package both the function and the target instance into one callable object.


# Best-practice checklist

When working with instance methods and dynamic method behavior:

- Use `self` as the first parameter name by convention.
- Prefer normal class definitions over monkey-patching unless runtime modification is genuinely needed.
- If you need a callback tied to an object, storing a bound method is often clean and expressive.
- Do not use repeated `obj.method is obj.method` as a test for logical method equivalence.
- For debugging, inspect `.__func__` and `.__self__`.
- Be careful when assigning attributes to an instance using names that already exist on the class.
- A plain function placed directly in an instance dictionary does not get normal class-function binding behavior.
- Use `types.MethodType` when manual per-instance binding is intentional.
- Understand that a bound method can keep its instance alive as long as the bound method itself remains referenced.
- Prefer clear, explicit designs over clever runtime patching in production code.
- Use assertions and small experiments to verify object-model assumptions.


# Optional challenge extensions

These are deliberately left without full solutions so you can continue practicing:

1. Implement an event system that stores **weak references** to bound-method instances so subscribers do not remain alive only because of subscriptions.
2. Build a descriptor that counts how many times each instance invokes a wrapped method.
3. Create a method registry that serializes only a method's qualified name and an object identifier, then discuss why restoring it safely is difficult.
4. Compare ordinary instance methods with `staticmethod` and `classmethod`.
5. Investigate how `super()` obtains a bound method.
6. Write a test utility that detects accidental method shadowing in an object's `__dict__`.
7. Implement a descriptor that refuses binding for instances of the wrong type.
8. Measure the difference between repeatedly looking up `obj.method()` and storing `method = obj.method` inside a tight loop. Treat the result as an implementation/performance experiment, not a language guarantee.
9. Explore whether custom callable descriptors need to implement `__set_name__`.
10. Build a plugin-style command dispatcher where runtime-added class functions automatically become instance-bound commands.
